In [2]:
import math
import importlib
import load_data_from_cache as cache_loader
import CNN_BiLSTM_src as cbl
from torch.utils.data import DataLoader
import torch

In [3]:
train_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/train")
train_emb_loader = DataLoader(train_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
val_emb_ds = cache_loader.CachedEmbeddingDataset('cache/esm2_t30/val')
val_emb_loader = DataLoader(val_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
test_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/test")
test_emb_loader = DataLoader(test_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)

In [5]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
model = cbl.CNN_BiLSTM_Head(
    d_in=640,
    cnn_hidden=32,
    cnn_layers=1,
    cnn_kernel=3,
    lstm_hidden=32,
    lstm_layers=1,
    mlp_hidden=16,
    dropout=0.1,
    use_layernorm=True,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

for epoch in range(1, 21):
    tr = cbl.train_one_epoch(model, train_emb_loader, optimizer, device)
    va_loss, va_r = cbl.eval_one_epoch(model, val_emb_loader, device, clamp_for_metrics=True)
    print(f"epoch {epoch:02d} | train {tr:.5f} | val {va_loss:.5f} | val pearson {va_r:.4f}")

epoch 01 | train 0.04218 | val 0.03520 | val pearson 0.7057


KeyboardInterrupt: 